# Course 6 - Viz Capstone + SciPy Bridge

Seaborn for statistical plots, SciPy for distributions and optimization,
then the **Grand Finale Lab** - your end-to-end Week 1 capstone.

**Sections**
1. Seaborn for statistical plotting (0:00-0:30)
2. SciPy stats (0:30-1:00)
3. SciPy optimize + Grand Finale (1:00-1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
from shared.data_utils import load_dataset
rng = np.random.default_rng(0)
sns.set_theme(style='whitegrid')


## 1. Seaborn - statistical plots in one line

In [ ]:
penguins = load_dataset('penguins').dropna()
penguins.head()


### Scatter coloured by category

In [ ]:
sns.scatterplot(data=penguins, x='bill_length_mm', y='bill_depth_mm',
                hue='species', style='sex');


### Boxplot - distribution per group

In [ ]:
sns.boxplot(data=penguins, x='species', y='body_mass_g', hue='sex');


### Violinplot - boxplot + KDE in one

In [ ]:
sns.violinplot(data=penguins, x='species', y='flipper_length_mm');


### Pairplot - every numeric column against every other

In [ ]:
sns.pairplot(penguins, hue='species', height=1.8);


### Heatmap - correlation matrix

In [ ]:
corr = penguins.select_dtypes('number').corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0);


## 2. SciPy stats - distributions and tests

### PDFs and CDFs

In [ ]:
x = np.linspace(-4, 4, 200)
fig, ax = plt.subplots()
for mu, sigma in [(0, 1), (0, 0.5), (1, 1)]:
    ax.plot(x, stats.norm.pdf(x, mu, sigma), label=f'N({mu}, {sigma})')
ax.set_title('Normal PDFs')
ax.legend();


In [ ]:
fig, ax = plt.subplots()
ax.plot(x, stats.norm.cdf(x), label='Normal CDF')
ax.plot(x, stats.t.cdf(x, df=3), label='t(df=3) CDF')
ax.plot(x, stats.chi2.cdf(np.clip(x, 0, None), df=3), label='chi2(df=3) CDF')
ax.legend();


### Fit a Normal to a sample

In [ ]:
samples = rng.normal(loc=2.0, scale=1.5, size=1000)
mu_hat, sigma_hat = stats.norm.fit(samples)
print(f'mu_hat = {mu_hat:.3f}, sigma_hat = {sigma_hat:.3f}')

xs = np.linspace(samples.min(), samples.max(), 200)
fig, ax = plt.subplots()
ax.hist(samples, bins=30, density=True, alpha=0.5)
ax.plot(xs, stats.norm.pdf(xs, mu_hat, sigma_hat), lw=2, label='fitted')
ax.legend();


### One-sample t-test

In [ ]:
res = stats.ttest_1samp(samples, popmean=0)
print('t =', res.statistic, '  p =', res.pvalue)


## 3. SciPy optimize - minimize a function

### Rosenbrock - the classical test bed

In [ ]:
def rosen(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

res = optimize.minimize(rosen, x0=[0.0, 0.0])
print('x* =', res.x.round(4))
print('f* =', res.fun)
print('iters:', res.nit)


### A 1-D loss surface - what every model trainer sees

In [ ]:
f = lambda x: (x - 3)**2 + np.sin(5 * x)
xs = np.linspace(-2, 8, 400)
fig, ax = plt.subplots()
ax.plot(xs, f(xs))
res = optimize.minimize(f, x0=0.0)
ax.axvline(res.x[0], color='red', ls='--', label=f'min at x={res.x[0]:.3f}')
ax.legend(); ax.set_title('A loss surface');


## Grand Finale Lab - end-to-end pipeline

**Mission.** Load `stock-prices`, inject synthetic NaNs, clean with Pandas, compute a 7-day rolling mean with NumPy, plot raw + smoothed with Matplotlib. This is the Week 1 capstone - every skill from the 6 courses fires in 6 cells.

In [ ]:
# Step 1 - load and inject NaNs
df = load_dataset('stock-prices').copy()
DATE  = df.columns[0]
PRICE = df.columns[1]
mask = rng.random(len(df)) < 0.05
df.loc[mask, PRICE] = np.nan
print('shape:', df.shape, '  NaNs injected:', int(mask.sum()))


In [ ]:
# Step 2 - parse dates and sort
df[DATE] = pd.to_datetime(df[DATE])
df = df.sort_values(DATE).reset_index(drop=True)


In [ ]:
# Step 3 - clean: forward-fill the price
df[PRICE] = df[PRICE].ffill()
assert df[PRICE].isna().sum() == 0


In [ ]:
# Step 4 - NumPy 7-day moving average via convolution
window = 7
kernel = np.ones(window) / window
ma = np.convolve(df[PRICE].values, kernel, mode='valid')
ma_full = np.concatenate([np.full(window - 1, np.nan), ma])
print('ma length matches df:', len(ma_full) == len(df))


In [ ]:
# Step 5 - plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df[DATE], df[PRICE], label='price', alpha=0.5)
ax.plot(df[DATE], ma_full, label='7-day MA', lw=2)
ax.set_xlabel('date'); ax.set_ylabel(PRICE)
ax.set_title('AAPL - closing price + 7-day moving average')
ax.legend(); fig.autofmt_xdate();


### Recap - Week 1 in one breath
* Seaborn for fast statistical plots; SciPy stats for distributions; SciPy optimize for the loss landscape every model walks.
* The Grand Finale = load -> clean -> smooth -> plot. That is the shape of every analysis you will write.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

# Exercise 1 - Fit a Normal to `tips`

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from shared.data_utils import load_dataset
tips = load_dataset('tips')


**Task 1.** Use `stats.norm.fit` to estimate `(mu, sigma)` from `tips['total_bill']`.

In [ ]:
# your code here


**Task 2.** Plot a density histogram of `total_bill` and overlay the fitted PDF.

In [ ]:
# your code here


**Task 3.** Report whether the fit looks good. (Eyeball - Course 4 of Week 4 covers formal tests.)

In [ ]:
# your code here


### Exercise 1 - Solution

# Exercise 1 - Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from shared.data_utils import load_dataset
tips = load_dataset('tips')
vals = tips['total_bill'].values


**Task 1.**

In [ ]:
mu, sigma = stats.norm.fit(vals)
print(f'mu = {mu:.3f}, sigma = {sigma:.3f}')


**Task 2.**

In [ ]:
xs = np.linspace(vals.min(), vals.max(), 200)
fig, ax = plt.subplots()
ax.hist(vals, bins=30, density=True, alpha=0.5)
ax.plot(xs, stats.norm.pdf(xs, mu, sigma), lw=2, label='fitted Normal')
ax.set_xlabel('total_bill'); ax.legend();


**Task 3.**

In [ ]:
# The distribution is right-skewed; a Normal underfits the right tail.
# A log-Normal or Gamma would likely do better.


---

## Exercise 2

# Exercise 2 - Minimize a 1-D function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize


**Task 1.** Define `f(x) = (x - 3)**2 + 5`.

In [ ]:
# your code here


**Task 2.** Use `optimize.minimize` starting from `x0 = 0.0`. Report `res.x` and `res.fun`.

In [ ]:
# your code here


**Task 3.** Verify by computing `f(res.x)` and checking it equals `res.fun` up to floating-point tolerance.

In [ ]:
# your code here


**Task 4 (stretch).** Plot `f` over `x in [-2, 8]` and mark the minimum with a vertical line.

In [ ]:
# your code here


### Exercise 2 - Solution

# Exercise 2 - Solutions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize


**Task 1.**

In [ ]:
def f(x):
    return (x - 3)**2 + 5


**Task 2.**

In [ ]:
res = optimize.minimize(f, x0=0.0)
print('x* =', res.x[0])
print('f* =', res.fun)


**Task 3.**

In [ ]:
assert np.isclose(float(f(res.x[0])), res.fun)
print('verified - minimum at (3, 5)')


**Task 4.**

In [ ]:
xs = np.linspace(-2, 8, 200)
fig, ax = plt.subplots()
ax.plot(xs, f(xs))
ax.axvline(res.x[0], color='red', ls='--', label=f'min at x={res.x[0]:.3f}')
ax.legend();


---

## Exercise 3

# Exercise 3 - Grand Finale, reusable

**Mission.** Wrap the Grand Finale pipeline in a one-screen function and call it.

Constraints:
- The function should accept a DataFrame, a date-column name, a price-column name, and a window size.
- It returns the figure so callers can save or display it.
- It performs: parse date -> sort -> forward-fill -> moving average -> plot.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
rng = np.random.default_rng(0)

df = load_dataset('stock-prices').copy()
# inject 5% NaNs in the price column
PRICE = df.columns[1]
mask = rng.random(len(df)) < 0.05
df.loc[mask, PRICE] = np.nan


**Step 1.** Write `def grand_finale(df, date_col, price_col, window=7) -> plt.Figure`.

In [ ]:
# your code here


**Step 2.** Call it on the data and display the figure.

In [ ]:
# your code here


**Step 3 (stretch).** Add a `title` parameter; default it to `f'{price_col} + {window}-day MA'`.

In [ ]:
# your code here


### Exercise 3 - Solution

# Exercise 3 - Solutions

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shared.data_utils import load_dataset
rng = np.random.default_rng(0)

df = load_dataset('stock-prices').copy()
DATE  = df.columns[0]
PRICE = df.columns[1]
mask = rng.random(len(df)) < 0.05
df.loc[mask, PRICE] = np.nan


**Step 1.**

In [ ]:
def grand_finale(df, date_col, price_col, window=7, title=None):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    df[price_col] = df[price_col].ffill()
    kernel = np.ones(window) / window
    ma = np.convolve(df[price_col].values, kernel, mode='valid')
    ma_full = np.concatenate([np.full(window - 1, np.nan), ma])
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df[date_col], df[price_col], label='price', alpha=0.5)
    ax.plot(df[date_col], ma_full, label=f'{window}-day MA', lw=2)
    ax.set_xlabel('date'); ax.set_ylabel(price_col)
    ax.set_title(title or f'{price_col} + {window}-day MA')
    ax.legend(); fig.autofmt_xdate()
    return fig


**Step 2.**

In [ ]:
fig = grand_finale(df, DATE, PRICE)
fig;


**Step 3.**

In [ ]:
fig = grand_finale(df, DATE, PRICE, window=14, title='AAPL - 14-day smoothing')
fig;
